# CS604 Colab launcher — Ana: controlled PEFT comparison (all 10,000 classes)

Fixes the gaps in the Task 2 method survey (report draft Section 8.1): every method at **one resolution (336)**, **one recipe** (`configs/peft_controlled_336.yaml`), **one view** (plain crop 0.9), **equal training minutes (20)**, scored on **one 10,000-class holdout** (5 per class, subset seed 0). Spec: `configs/ablation_peft_controlled.yaml`.

| Arm | Method | Backbone % |
|---|---|---|
| A | LoRA r24, qkv/proj/fc1/fc2, all 12 blocks | 4.12 |
| B | LoRA r40, blocks 4–11 only (concentrated) | 4.57 |
| C | AdaptFormer w192 (paper version) | 4.13 |
| D | Serial adapter w96 | 4.14 |
| E | BitFit | 0.12 |
| F | Last-block attention | 2.75 |
| G | Prefix tuning, length 50 | 1.07 |
| A_seed1/2, C_seed1/2 | seeds for A and C | |

About 25 min per arm on an **A100** (do not mix GPU types: arms get equal minutes). All 11 arms ≈ 4.6 h, so split them over two sessions with `ONLY` below. Start session 2 after session 1 has written `prototypes_pilot_fit.pt` (or copy it into `CS604_RUNS/ablation_ablation_peft_controlled/` from `ablation_allclass_holdout_lora_no_extras/` first). If a session drops, Run all again: finished arms are skipped.

Runtime: **A100 GPU**. Use **Runtime → Run all**.


In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "ana-branch"
METHOD = "lora_r24_336_recipe"  # base config; only used to build prototypes_336.pt if it is missing
PROTO_PT = f"{DRIVE_OUT}/prototypes_336.pt"
SPEC = "configs/ablation_peft_controlled.yaml"
# Session 1: ["A_lora_r24", "C_adaptformer_w192", "D_adapter_w96", "B_lora_r40_blocks4_11", "A_seed1", "C_seed1"]
# Session 2: ["E_bitfit", "F_lastblock_attention", "G_prefix_p50", "A_seed2", "C_seed2"]
ONLY = []  # empty = every arm
PILOT_DATA_DIR = "/content/pilot_data"
FEATURE_DIR = "/content/features_pilot"  # local disk
# End the runtime after the last cell, or as soon as training fails.
AUTO_DISCONNECT = True


def end_runtime(reason):
    """Flush Drive writes, then release the GPU so it stops using compute units."""
    print(reason, flush=True)
    if not AUTO_DISCONNECT:
        return
    from google.colab import drive, runtime
    try:
        drive.flush_and_unmount()  # without this the last checkpoint can be lost
        print("Drive flushed and unmounted", flush=True)
    except Exception as error:
        print(f"Drive flush failed: {error}", flush=True)
    runtime.unassign()

## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Download data to `/content/data`

In [ ]:
!python scripts/download_data.py --dest {DATA_DIR}

## 4b. Build class prototypes at 336 (once)

Same as Van's prototype head, but built from 336 features (built with `configs/{METHOD}.yaml`, so at its `img_size`). Run **once**; about 20–25 min on an A100. Takes a while (embeds train images). Skip if the file already exists on Drive.

In [ ]:
from pathlib import Path

if Path(PROTO_PT).is_file():
    print("Already have", PROTO_PT)
else:
    !python scripts/build_prototypes.py --config configs/{METHOD}.yaml --data_root {DATA_DIR} --output {PROTO_PT}

## 5. Ablation

In [ ]:
import subprocess
import sys

proc = subprocess.run([sys.executable, "-u", "scripts/ablation.py", "--spec", SPEC,
                       "--data_root", DATA_DIR, "--output_dir", DRIVE_OUT,
                       "--pilot_data_dir", PILOT_DATA_DIR, "--feature_dir", FEATURE_DIR]
                      + (["--only", *ONLY] if ONLY else []))
if proc.returncode != 0:
    end_runtime(f"ablation failed (exit {proc.returncode}); see the output above")
    raise SystemExit("ablation failed")

## 6. Results

`delta_vs_A_points`: change in pilot holdout top-1 (percentage points) against the full recipe. Negative = turning that piece off hurts, so that piece helped. Treat differences under ~0.3 points as ties (one seed).

In [ ]:
import json
from pathlib import Path

import pandas as pd

# Read the per-arm markers, not ablation.csv: with ONLY, each session rewrites ablation.csv
# with its own arms, while arms/*.json holds every finished arm from both sessions.
arms_dir = Path(DRIVE_OUT) / "ablation_ablation_peft_controlled" / "arms"
table = pd.DataFrame([json.loads(p.read_text()) for p in sorted(arms_dir.glob("*.json"))])
ref = table.loc[table["arm"] == "A_lora_r24", "final_val_acc"]
if len(ref):
    table["delta_vs_A_points"] = (100 * (table["final_val_acc"] - ref.iloc[0])).round(2)
display(table[["arm", "changes", "final_val_acc", "delta_vs_A_points", "final_val_loss",
               "backbone_pct", "peak_vram_gb", "training_min", "wall_min"]])


## 7. End the runtime

In [ ]:
end_runtime("all cells finished; ending the runtime")